# Treino do `.task` de Libras sem Model Maker (Python 3.12)

Este notebook usa o **MediaPipe e o TensorFlow atuais do Colab**. Não precisa do Model Maker nem do Python 3.10.

**Como funciona**
1. **Achar a mão e calcular o resumo de 128 números** de cada foto, com o MediaPipe atual. Roda em paralelo em todos os núcleos do processador. Antes, ele testa a GPU e só usa se ela for mais rápida. O resultado fica salvo no Drive, então essa parte é feita uma vez só.
2. **Treinar a etapa final** (BatchNorm → ReLU → camada densa), a mesma arquitetura do Model Maker, em Keras. Usa a GPU se houver.
3. **Gerar o `.task`**: os pesos novos são gravados dentro do classificador de um `.task` modelo (o do projeto, de 2025), com as mesmas 21 letras. O resto do pacote do Google não muda.
4. **Conferir** o `.task` novo no reconhecedor do MediaPipe, do jeito que o app usa.

**Treino sem vazamento:** o treino usa só a pasta `train` (90% treino, 10% validação). A pasta `test` só entra no fim, para avaliar. Lote de 32, 100 épocas, espelhamento nos pontos da mão (igual ao RF, SVM e SNN) e semente fixa.

**Ambiente de execução:** no Colab Pro, use **CPU com "RAM alta"** ou outra opção com mais núcleos. A célula 2 mostra quantos núcleos a máquina tem.

**Como usar:** `Ambiente de execução > Executar tudo`. No fim, ele baixa `treino_task_libras.zip`.

## 1. Configuração

In [ ]:
# De onde vêm as fotos:
#   "kaggle" -> baixa a base williansoliveira/libras (46.262 fotos). Se o Kaggle pedir login, use "drive" ou "zip".
#   "drive"  -> usa o libras.zip do seu Google Drive (caminho em ZIP_DRIVE).
#   "zip"    -> usa um libras.zip que você sobe no painel de arquivos do Colab.
FONTE = "kaggle"
ZIP_DRIVE = "/content/drive/MyDrive/libras.zip"
FRACAO = 1.0       # 1.0 = base inteira; 0.1 = cerca de 10% de cada letra

# .task modelo (precisa ter as 21 letras): o do seu Space no Hugging Face. Se não baixar,
# coloque o seu gesture_recognizer.task no Drive (caminho abaixo) ou suba no painel de arquivos.
TASK_MODELO_URL = "https://huggingface.co/spaces/RubensMota13/LibrasMediaPipe/resolve/main/models/gesture_recognizer.task"
TASK_MODELO_DRIVE = "/content/drive/MyDrive/gesture_recognizer.task"

PASTA_DRIVE = "/content/drive/MyDrive/libras_treino"   # onde guardar o resumo das fotos e os resultados
EPOCAS = 100
LOTE = 32
ESPELHAR = True
SEMENTE = 42
TENTAR_GPU = True

## 2. Preparar o ambiente

In [ ]:
!apt-get -qq install -y libegl1 libgles2 > /dev/null 2>&1
!pip -q install mediapipe ai-edge-litert kagglehub
import os, subprocess
import mediapipe as mp, tensorflow as tf
print("MediaPipe", mp.__version__, "| TensorFlow", tf.__version__)
print("Núcleos do processador:", os.cpu_count())
print("GPU para o treino:", tf.config.list_physical_devices("GPU") or "nenhuma (o treino é rápido mesmo assim)")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(PASTA_DRIVE, exist_ok=True)

## 3. Código do treino

In [ ]:
%%writefile /content/libras_task.py
"""Treino do .task de Libras sem o Model Maker (Python 3.12, MediaPipe e TensorFlow atuais).

Etapas:
  1. extrair(): acha a mão em cada foto com o MediaPipe atual, em paralelo em todos os
     núcleos (ou na GPU, se o MediaPipe conseguir usá-la), e calcula o resumo de 128
     números com o gesture_embedder do próprio .task. Resultado salvo num .npz.
  2. treinar(): treina a etapa final (BatchNorm -> ReLU -> Dropout -> Dense + softmax),
     a mesma arquitetura que o Model Maker usa, em Keras (usa a GPU se houver).
  3. montar_task(): coloca os pesos novos dentro do classificador de um .task modelo
     (mesma arquitetura e mesmas letras), sem mexer no resto do pacote.
  4. conferir_task(): roda o .task novo no GestureRecognizer do MediaPipe, como o app faz.
"""
import io
import os
import time
import zipfile
from multiprocessing import Pool

import numpy as np

EXT = (".jpg", ".jpeg", ".png")
LETRAS_TASK = None  # rótulos do .task modelo, lidos em ler_modelo()
_PASTA_TMP = "/tmp/libras_task"


# ---------------------------------------------------------------- .task modelo
def ler_modelo(caminho_task):
    """Extrai do .task modelo: hand_landmarker.task, gesture_embedder.tflite,
    custom_gesture_classifier.tflite e a lista de rótulos (none + letras)."""
    os.makedirs(_PASTA_TMP, exist_ok=True)
    with zipfile.ZipFile(caminho_task) as ext:
        open(f"{_PASTA_TMP}/hand_landmarker.task", "wb").write(ext.read("hand_landmarker.task"))
        interno = zipfile.ZipFile(io.BytesIO(ext.read("hand_gesture_recognizer.task")))
        nomes = interno.namelist()
        if "custom_gesture_classifier.tflite" not in nomes:
            raise ValueError("Este .task não tem classificador próprio (custom_gesture_classifier.tflite). "
                             "Use o .task do projeto (o de 2025), não o padrão do Google.")
        for n in nomes:
            open(f"{_PASTA_TMP}/{n}", "wb").write(interno.read(n))
    cls = open(f"{_PASTA_TMP}/custom_gesture_classifier.tflite", "rb").read()
    rotulos = zipfile.ZipFile(io.BytesIO(cls)).read("labels.txt").decode().split()
    return rotulos


# ---------------------------------------------------------------- 1. extração
_landmarker = None


def _iniciar(usar_gpu):
    global _landmarker
    import mediapipe as mp
    from mediapipe.tasks.python import BaseOptions, vision
    base = BaseOptions(model_asset_path=f"{_PASTA_TMP}/hand_landmarker.task",
                       delegate=BaseOptions.Delegate.GPU if usar_gpu else BaseOptions.Delegate.CPU)
    _landmarker = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=base, num_hands=1, running_mode=vision.RunningMode.IMAGE))
    _ = mp


def _achar_mao(caminho):
    import mediapipe as mp
    try:
        r = _landmarker.detect(mp.Image.create_from_file(caminho))
    except Exception:
        return None
    if not r.hand_landmarks:
        return None
    h = r.handedness[0][0]
    direita = h.score if h.category_name == "Right" else 1.0 - h.score
    return ([[p.x, p.y, p.z] for p in r.hand_landmarks[0]],
            [[p.x, p.y, p.z] for p in r.hand_world_landmarks[0]], direita)


def _medir(usar_gpu, amostra, fila):
    try:
        _iniciar(usar_gpu)
        _achar_mao(amostra[0])
        t = time.time()
        for c in amostra:
            _achar_mao(c)
        fila.put(time.time() - t)
    except Exception as e:  # noqa: BLE001
        fila.put(f"{type(e).__name__}: {e}"[:150])


def _testar_gpu(amostra, limite_s=90):
    """Mede a GPU do MediaPipe num processo separado (ela pode travar ou falhar em
    máquinas sem driver gráfico). Só usa a GPU se ela ganhar de todos os núcleos juntos."""
    import multiprocessing as mpc
    tempos = {}
    for nome, gpu in (("GPU", True), ("1 núcleo do processador", False)):
        fila = mpc.Queue()
        proc = mpc.Process(target=_medir, args=(gpu, amostra, fila))
        proc.start()
        proc.join(limite_s)
        if proc.is_alive():
            proc.kill()
            tempos[nome] = "travou"
        else:
            tempos[nome] = fila.get() if not fila.empty() else "falhou"
    print("Teste em", len(amostra), "fotos:", {k: (f"{v:.1f}s" if isinstance(v, float) else v) for k, v in tempos.items()})
    g, c = tempos["GPU"], tempos["1 núcleo do processador"]
    if not isinstance(g, float):
        print("GPU do MediaPipe não funcionou aqui; usando o processador.")
        return False
    if not isinstance(c, float):
        return True
    return g * os.cpu_count() < c


def listar_fotos(raiz):
    fotos = []
    for parte in ("train", "test"):
        for letra in sorted(os.listdir(os.path.join(raiz, parte))):
            d = os.path.join(raiz, parte, letra)
            if not os.path.isdir(d) or letra.lower() == "none":
                continue
            for f in sorted(os.listdir(d)):
                if f.lower().endswith(EXT):
                    fotos.append((parte, letra.upper(), os.path.join(d, f)))
    return fotos


def resumir(pontos, mundo, direita):
    """[N,21,3], [N,21,3], [N] -> [N,128] com o gesture_embedder do .task modelo."""
    from ai_edge_litert.interpreter import Interpreter
    rede = Interpreter(model_path=f"{_PASTA_TMP}/gesture_embedder.tflite")
    rede.allocate_tensors()
    ent = {d["name"]: d["index"] for d in rede.get_input_details()}
    sai = rede.get_output_details()[0]["index"]
    out = np.empty((len(pontos), 128), np.float32)
    for i in range(len(pontos)):
        rede.set_tensor(ent["hand"], np.asarray(pontos[i], np.float32)[None])
        rede.set_tensor(ent["handedness"], np.array([[direita[i]]], np.float32))
        rede.set_tensor(ent["world_hand"], np.asarray(mundo[i], np.float32)[None])
        rede.invoke()
        out[i] = rede.get_tensor(sai)[0]
    return out


def espelhar(pontos, mundo, direita):
    p = np.array(pontos, np.float32)
    m = np.array(mundo, np.float32)
    p[:, :, 0] = 1.0 - p[:, :, 0]
    m[:, :, 0] *= -1
    return p, m, 1.0 - np.asarray(direita, np.float32)


def extrair(raiz, saida_npz, tentar_gpu=True, nucleos=None):
    fotos = listar_fotos(raiz)
    caminhos = [f[2] for f in fotos]
    nucleos = nucleos or os.cpu_count()
    print(f"{len(fotos)} fotos | núcleos do processador: {os.cpu_count()}")
    usar_gpu = tentar_gpu and _testar_gpu(caminhos[:30])
    t0 = time.time()
    res = []
    n = len(caminhos)
    passo = max(1, n // 100)

    def progresso(i):
        if i % passo == 0 or i == n:
            gasto = time.time() - t0
            falta = gasto / i * (n - i)
            print(f"  Achando mãos: {i}/{n} ({100 * i / n:.0f}%) | passou {gasto / 60:.1f} min | "
                  f"falta ~{falta / 60:.1f} min", flush=True)

    if usar_gpu:
        print("Usando a GPU do MediaPipe.")
        _iniciar(True)
        for i, c in enumerate(caminhos, 1):
            res.append(_achar_mao(c))
            progresso(i)
    else:
        print(f"Usando o processador em paralelo ({nucleos} processos).")
        with Pool(nucleos, initializer=_iniciar, initargs=(False,)) as pool:
            for i, r in enumerate(pool.imap(_achar_mao, caminhos, chunksize=16), 1):
                res.append(r)
                progresso(i)
    ok = [i for i, r in enumerate(res) if r is not None]
    pontos = np.array([res[i][0] for i in ok], np.float32)
    mundo = np.array([res[i][1] for i in ok], np.float32)
    direita = np.array([res[i][2] for i in ok], np.float32)
    print("Calculando o resumo de 128 números...")
    emb = resumir(pontos, mundo, direita)
    pe, me, de = espelhar(pontos, mundo, direita)
    emb_esp = resumir(pe, me, de)
    np.savez_compressed(
        saida_npz, emb=emb, emb_esp=emb_esp,
        y=np.array([fotos[i][1] for i in ok]), parte=np.array([fotos[i][0] for i in ok]),
        arquivo=np.array([os.path.relpath(fotos[i][2], raiz) for i in ok]),
        sem_mao=np.array([os.path.relpath(fotos[i][2], raiz) for i, r in enumerate(res) if r is None]),
        tempo_min=(time.time() - t0) / 60, dispositivo="GPU" if usar_gpu else f"CPU x{nucleos}")
    print(f"Pronto: {len(ok)} fotos com mão, {n - len(ok)} sem mão, em {(time.time() - t0) / 60:.1f} min.")


# ---------------------------------------------------------------- 2. treino
def treinar(npz, rotulos, epocas=100, lote=32, lr=0.001, espelhar_treino=True, semente=42, dropout=0.05):
    import tensorflow as tf
    import keras
    keras.utils.set_random_seed(semente)
    d = np.load(npz)
    idx = {r: i for i, r in enumerate(rotulos)}
    y = np.array([idx[v] for v in d["y"]])
    tr, te = d["parte"] == "train", d["parte"] == "test"
    rng = np.random.default_rng(semente)
    ids = rng.permutation(np.where(tr)[0])
    n_val = len(ids) // 10
    val, trn = ids[:n_val], ids[n_val:]
    x_tr, y_tr = d["emb"][trn], y[trn]
    if espelhar_treino:
        x_tr = np.concatenate([x_tr, d["emb_esp"][trn]])
        y_tr = np.concatenate([y_tr, y[trn]])
    k = len(rotulos)
    oh = lambda v: keras.utils.to_categorical(v, k)

    ent = keras.Input(shape=(128,), name="hand_embedding")
    x = keras.layers.BatchNormalization()(ent)
    x = keras.layers.ReLU()(x)
    x = keras.layers.Dropout(dropout)(x)
    sai = keras.layers.Dense(k, activation="softmax", name="custom_gesture_recognizer_out")(x)
    modelo = keras.Model(ent, sai)
    passos = int(np.ceil(len(x_tr) / lote))
    agenda = keras.optimizers.schedules.ExponentialDecay(lr, decay_steps=passos, decay_rate=0.99, staircase=True)
    modelo.compile(optimizer=keras.optimizers.Adam(agenda), loss="categorical_crossentropy",
                   metrics=["categorical_accuracy"])
    print(f"Treino: {len(x_tr)} exemplos{' (com espelhados)' if espelhar_treino else ''} | "
          f"validação: {len(val)} | teste: {te.sum()} | dispositivo: "
          f"{'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'}")
    t0 = time.time()
    melhor = keras.callbacks.ModelCheckpoint("/tmp/libras_task/melhor.weights.h5", monitor="val_categorical_accuracy",
                                             save_best_only=True, save_weights_only=True)
    hist = modelo.fit(x_tr, oh(y_tr), batch_size=lote, epochs=epocas, shuffle=True,
                      validation_data=(d["emb"][val], oh(y[val])), callbacks=[melhor], verbose=2)
    modelo.load_weights("/tmp/libras_task/melhor.weights.h5")
    t_treino = (time.time() - t0) / 60

    prob = modelo.predict(d["emb"][te], verbose=0)
    pred, real = prob.argmax(1), y[te]
    matriz = np.zeros((k, k), int)
    for a, b in zip(real, pred):
        matriz[a, b] += 1
    por_letra = {}
    for i, r in enumerate(rotulos):
        sup = matriz[i].sum()
        if not sup:
            continue
        vp = matriz[i, i]
        p = vp / matriz[:, i].sum() if matriz[:, i].sum() else 0.0
        rv = vp / sup
        por_letra[r] = {"precisao": round(float(p), 4), "revocacao": round(float(rv), 4),
                        "f1": round(float(2 * p * rv / (p + rv)) if p + rv else 0.0, 4), "fotos": int(sup)}
    pesos = np.array([v["fotos"] for v in por_letra.values()])
    media = {m: round(float(np.average([v[m] for v in por_letra.values()], weights=pesos)), 4)
             for m in ("precisao", "revocacao", "f1")}
    acc = float((pred == real).mean())
    print(f"\nTESTE (pasta test, nunca vista no treino): acurácia {acc:.4f} | {media}")
    res = {
        "config": {"epocas": epocas, "lote": lote, "lr": lr, "espelhar": espelhar_treino, "semente": semente,
                   "dropout": dropout, "divisao": "train -> 90% treino / 10% validação; test só para avaliar",
                   "extracao": str(d["dispositivo"])},
        "fotos": {"treino": int(len(trn)), "validacao": int(len(val)), "teste": int(te.sum()),
                  "sem_mao": int(len(d["sem_mao"]))},
        "teste": {"acuracia": round(acc, 4), **media},
        "por_letra": por_letra, "rotulos": list(rotulos), "matriz_confusao": matriz.tolist(),
        "historico": {kk: [float(v) for v in vv] for kk, vv in hist.history.items()},
        "tempo_min": {"extracao": round(float(d["tempo_min"]), 1), "treino": round(t_treino, 1)},
    }
    return modelo, res


# ---------------------------------------------------------------- 3. .task
def _pesos_tflite(modelo):
    """BatchNorm vira MUL + ADD no .tflite; Dense vira FULLY_CONNECTED [saídas, 128]."""
    bn = [l for l in modelo.layers if l.__class__.__name__ == "BatchNormalization"][0]
    gamma, beta, media, var = [np.asarray(w, np.float32) for w in bn.get_weights()]
    mul = gamma / np.sqrt(var + bn.epsilon)
    add = beta - media * mul
    densa = modelo.get_layer("custom_gesture_recognizer_out")
    kernel, bias = [np.asarray(w, np.float32) for w in densa.get_weights()]
    return {"mul": mul, "sub": add, "MatMul": kernel.T.copy(), "BiasAdd": bias}


def _desligar_canned(dados_canned):
    """Zera o classificador de gestos prontos do Google (punho, mão aberta, joinha...),
    fazendo-o responder sempre "None". Sem isso, em algumas fotos o MediaPipe devolve o
    gesto pronto (nome vazio) no lugar da letra. O arquivo precisa existir no pacote."""
    from ai_edge_litert.interpreter import Interpreter
    dados = bytearray(dados_canned)
    it = Interpreter(model_content=bytes(dados))
    it.allocate_tensors()
    trocados = 0
    for t in it.get_tensor_details():
        nome = t["name"]
        if ";" in nome or not (nome.endswith("new_classifier/MatMul") or "new_classifier/BiasAdd/ReadVariableOp" in nome):
            continue
        antigo = it.get_tensor(t["index"]).astype(np.float32)
        novo = np.zeros_like(antigo)
        if antigo.ndim == 1:
            novo[0] = 50.0  # índice 0 = "None"
        b_ant = antigo.tobytes()
        pos = dados.find(b_ant)
        if pos < 0:
            raise RuntimeError(f"Não achei os pesos '{nome}' do classificador do Google.")
        dados[pos:pos + len(b_ant)] = novo.tobytes()
        trocados += 1
    if trocados != 2:
        raise RuntimeError(f"Esperava trocar 2 blocos no classificador do Google, troquei {trocados}.")
    return bytes(dados)


def montar_task(modelo, saida_task, desligar_gestos_google=False):
    """Copia o .task modelo e troca só os números do classificador próprio.

    O classificador do .task modelo tem a mesma arquitetura (BatchNorm + ReLU + Dense, 22 saídas),
    então os pesos novos têm exatamente o mesmo tamanho e são gravados por cima dos antigos,
    mantendo os rótulos e os metadados que o MediaPipe exige."""
    from ai_edge_litert.interpreter import Interpreter
    cls_path = f"{_PASTA_TMP}/custom_gesture_classifier.tflite"
    dados = bytearray(open(cls_path, "rb").read())
    it = Interpreter(model_path=cls_path)
    it.allocate_tensors()
    novos = _pesos_tflite(modelo)
    trocados = 0
    for t in it.get_tensor_details():
        chave = next((c for c in ("mul", "sub", "MatMul", "BiasAdd") if t["name"].endswith(c)
                      or t["name"].endswith(c + "/ReadVariableOp")), None)
        if chave is None or ";" in t["name"]:
            continue
        try:
            antigo = it.get_tensor(t["index"]).astype(np.float32)
        except ValueError:
            continue
        novo = novos[chave].reshape(antigo.shape)
        b_ant = antigo.tobytes()
        pos = dados.find(b_ant)
        if pos < 0 or dados.find(b_ant, pos + 1) >= 0:
            raise RuntimeError(f"Não achei de forma única os pesos '{t['name']}' no classificador modelo.")
        dados[pos:pos + len(b_ant)] = novo.tobytes()
        trocados += 1
    if trocados != 4:
        raise RuntimeError(f"Esperava trocar 4 blocos de pesos, troquei {trocados}.")
    # Remonta hand_gesture_recognizer.task e o .task final, como o Model Maker faz
    interno = io.BytesIO()
    with zipfile.ZipFile(interno, "w", zipfile.ZIP_STORED) as z:
        z.writestr("gesture_embedder.tflite", open(f"{_PASTA_TMP}/gesture_embedder.tflite", "rb").read())
        canned = open(f"{_PASTA_TMP}/canned_gesture_classifier.tflite", "rb").read()
        z.writestr("canned_gesture_classifier.tflite", _desligar_canned(canned) if desligar_gestos_google else canned)
        z.writestr("custom_gesture_classifier.tflite", bytes(dados))
    with zipfile.ZipFile(saida_task, "w", zipfile.ZIP_STORED) as z:
        z.writestr("hand_landmarker.task", open(f"{_PASTA_TMP}/hand_landmarker.task", "rb").read())
        z.writestr("hand_gesture_recognizer.task", interno.getvalue())
    print(f".task gerado: {saida_task} ({os.path.getsize(saida_task) / 1e6:.1f} MB)")


# ---------------------------------------------------------------- 4. conferência
def conferir_task(caminho_task, raiz, arquivos_teste, letras_esperadas, maximo=400, semente=0):
    """Roda o .task novo no GestureRecognizer (como o app) numa amostra do teste."""
    import mediapipe as mp
    from mediapipe.tasks.python import BaseOptions, vision
    g = vision.GestureRecognizer.create_from_options(vision.GestureRecognizerOptions(
        base_options=BaseOptions(model_asset_path=caminho_task)))
    rng = np.random.default_rng(semente)
    ids = rng.permutation(len(arquivos_teste))[:maximo]
    certo = errado = incerto = sem_mao = 0
    for i in ids:
        r = g.recognize(mp.Image.create_from_file(os.path.join(raiz, str(arquivos_teste[i]))))
        if not r.gestures:
            sem_mao += 1
            continue
        nome = r.gestures[0][0].category_name
        if nome == letras_esperadas[i]:
            certo += 1
        elif not nome or nome.lower() == "none":
            incerto += 1  # nenhuma letra passou de 50% de confiança
        else:
            errado += 1
    n = certo + errado + incerto
    print(f"Conferência no GestureRecognizer (como o app), {n} fotos de teste: {certo} certas ({certo / max(n, 1):.1%}), "
          f"{errado} letra errada, {incerto} sem letra acima de 50% de confiança.")
    return certo, errado, incerto


## 4. Fotos e `.task` modelo

In [ ]:
import os, random, shutil, zipfile, urllib.request, sys
sys.path.insert(0, "/content")
import libras_task as L

def achar_raiz(base):
    for raiz, pastas, _ in os.walk(base):
        if "train" in pastas and "test" in pastas:
            return raiz
    raise FileNotFoundError("Não achei as pastas train/ e test/ em " + base)

if FONTE == "kaggle":
    import kagglehub
    raiz = achar_raiz(kagglehub.dataset_download("williansoliveira/libras"))
else:
    arq_zip = ZIP_DRIVE if FONTE == "drive" else "/content/libras.zip"
    if not os.path.exists(arq_zip):
        raise FileNotFoundError(f"Não achei {arq_zip}.")
    shutil.rmtree("/content/libras_zip", ignore_errors=True)
    zipfile.ZipFile(arq_zip).extractall("/content/libras_zip")
    raiz = achar_raiz("/content/libras_zip")

# Fração das fotos (links, sem copiar)
DADOS = "/content/dados"
shutil.rmtree(DADOS, ignore_errors=True)
rng = random.Random(SEMENTE)
for parte in ("train", "test"):
    for letra in sorted(os.listdir(os.path.join(raiz, parte))):
        origem = os.path.join(raiz, parte, letra)
        if not os.path.isdir(origem) or letra.lower() == "none":
            continue
        fotos = sorted(f for f in os.listdir(origem) if f.lower().endswith(L.EXT))
        if FRACAO < 1.0:
            fotos = sorted(rng.sample(fotos, max(1, round(len(fotos) * FRACAO))))
        os.makedirs(os.path.join(DADOS, parte, letra), exist_ok=True)
        for f in fotos:
            os.symlink(os.path.join(origem, f), os.path.join(DADOS, parte, letra, f))
print("Pasta das fotos:", raiz)

# .task modelo
TASK_MODELO = "/content/modelo_2025.task"
if os.path.exists("/content/gesture_recognizer.task"):
    shutil.copy("/content/gesture_recognizer.task", TASK_MODELO)
elif os.path.exists(TASK_MODELO_DRIVE):
    shutil.copy(TASK_MODELO_DRIVE, TASK_MODELO)
else:
    urllib.request.urlretrieve(TASK_MODELO_URL, TASK_MODELO)
ROTULOS = L.ler_modelo(TASK_MODELO)
print("Rótulos do .task modelo:", ROTULOS)

## 5. Achar a mão e calcular o resumo (a parte demorada; feita uma vez só)
Se o arquivo do resumo já existir no Drive (de uma rodada anterior com a mesma `FRACAO`), esta etapa é pulada.

In [ ]:
NPZ = os.path.join(PASTA_DRIVE, f"resumo_libras_fracao{FRACAO}.npz")
if os.path.exists(NPZ):
    print("Usando o resumo já salvo:", NPZ)
else:
    L.extrair(DADOS, "/content/resumo.npz", tentar_gpu=TENTAR_GPU)
    shutil.copy("/content/resumo.npz", NPZ)
    print("Salvo no Drive:", NPZ)

## 6. Treinar a etapa final

In [ ]:
modelo, res = L.treinar(NPZ, ROTULOS, epocas=EPOCAS, lote=LOTE, espelhar_treino=ESPELHAR, semente=SEMENTE)
import json
os.makedirs("/content/resultado", exist_ok=True)
json.dump(res, open("/content/resultado/resultado_task.json", "w"), ensure_ascii=False, indent=1)
print("Teste:", res["teste"])
for letra, v in res["por_letra"].items():
    print(f"  {letra}: P={v['precisao']:.2f} R={v['revocacao']:.2f} F1={v['f1']:.2f} ({v['fotos']} fotos)")

## 7. Gráficos

In [ ]:
import matplotlib.pyplot as plt, numpy as np
h = res["historico"]; ep = range(1, len(h["loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, h["categorical_accuracy"], label="treino"); ax[0].plot(ep, h["val_categorical_accuracy"], label="validação")
ax[0].set_title("Acurácia por época"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(ep, h["loss"], label="treino"); ax[1].plot(ep, h["val_loss"], label="validação")
ax[1].set_title("Perda por época"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.savefig("/content/resultado/curvas_task.png", dpi=150); plt.show()

rot = res["rotulos"]; m = np.array(res["matriz_confusao"])
usar = [i for i, x in enumerate(rot) if x.lower() != "none"]
m = m[np.ix_(usar, usar)]; let = [rot[i] for i in usar]
fig, ax = plt.subplots(figsize=(10, 9)); ax.imshow(m, cmap="Blues")
ax.set_xticks(range(len(let)), let); ax.set_yticks(range(len(let)), let)
ax.set_xlabel("letra prevista"); ax.set_ylabel("letra verdadeira")
ax.set_title(f"Matriz de confusão do .task no teste (acurácia {res['teste']['acuracia']:.3f})")
for i in range(len(let)):
    for j in range(len(let)):
        if m[i, j]:
            ax.text(j, i, m[i, j], ha="center", va="center", fontsize=7, color="white" if m[i, j] > m.max() / 2 else "black")
plt.tight_layout(); plt.savefig("/content/resultado/matriz_confusao_task.png", dpi=150); plt.show()

## 8. Gerar e conferir o `.task`

In [ ]:
L.montar_task(modelo, "/content/resultado/gesture_recognizer.task")
d = np.load(NPZ); te = d["parte"] == "test"
L.conferir_task("/content/resultado/gesture_recognizer.task", DADOS, d["arquivo"][te], d["y"][te], maximo=1000)

## 9. Baixar

In [ ]:
with open("/content/resultado/lista_fotos.txt", "w") as f:
    f.write("\n".join(map(str, d["arquivo"])))
!cd /content/resultado && rm -f /content/treino_task_libras.zip && zip -q /content/treino_task_libras.zip gesture_recognizer.task resultado_task.json lista_fotos.txt curvas_task.png matriz_confusao_task.png
!cp /content/treino_task_libras.zip {PASTA_DRIVE}/
from google.colab import files
files.download("/content/treino_task_libras.zip")
print("Também salvo no Drive em", PASTA_DRIVE)